# 02b Fitting last year's volatility and the text together

The within-year ridge in `02-Linear-models.ipynb` works in two steps: fit the straight line
$y = a + b\,x$ (with $x$ = `logvol_before`) on the training window, then fit a ridge of the line's residual on the
text. Louis's network (`Louis/05-MLP-updated-v2.ipynb`) learns the weight on $x$ *at the same time* as the text,
and ranks companies better than any of our models. This notebook tests whether that difference is the reason.

The two steps are not the same as one joint fit when the text and $x$ are correlated. By the Frisch–Waugh–Lovell
theorem, the coefficient on the text in the joint regression of $y$ on $(x, X)$ equals the regression of
$y$'s residual on $X$'s residual, both after removing $x$. The two-step fit removes $x$ from $y$ only, not from
$X$, so the text cannot correct the slope $b$, and the part of the text that tracks last year's volatility is
partly wasted on refitting what the line already did.

Three models, all within year (each year's own averages removed, so the text can only change how companies are
ranked, never the year's level), all scored with `scoring.py` against the straight line:

| Model | Fit | What it isolates |
|---|---|---|
| Two-step (02) | Line on the window, then ridge of its within-year residual on the text | Reproduces 02's within-year ridge |
| Joint | Ridge of $y$ on the text and $x$ together, with $x$ not penalised | The joint fit |
| No text | The within-year slope of $y$ on $x$ alone | How much of the joint gain needs the text at all |

The penalty $\lambda$ is chosen on the last training year and the model refitted on the whole window, as in 02.
Every prediction is the straight line's average for the test year plus the model's within-year part, so all three
have exactly the line's level and differ only in the spread. It runs on a CPU in a few minutes.

On Colab, the first cell connects Google Drive, gets the repo and copies `data.pkl`, as in 02. Locally it does nothing.

In [1]:
import importlib.metadata, importlib.util, os, shutil, subprocess, sys
from pathlib import Path

NOTEBOOK_DIR = "Alex"
IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB:
    if int(importlib.metadata.version("pandas").split(".")[0]) < 3:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas==3.0.6"], check=True)
        if "pandas" in sys.modules:
            print("pandas 3 installed but the old one was already loaded: restarting the runtime. Run all again.")
            os.kill(os.getpid(), 9)
    from google.colab import drive
    drive.mount("/content/drive")
    repo = Path("/content/dst-sprint2")
    if repo.exists():
        subprocess.run(["git", "-C", str(repo), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "https://github.com/Oscar339/dst-sprint2.git", str(repo)], check=True)
    processed = repo / "data" / "processed"
    processed.mkdir(parents=True, exist_ok=True)
    if not (processed / "data.pkl").exists():
        shutil.copy(Path("/content/drive/MyDrive/sprint2-data/data.pkl"), processed / "data.pkl")
    os.chdir(repo / NOTEBOOK_DIR)

In [2]:
import pickle, platform, time
sys.path.insert(0, ".")

import numpy as np
import pandas as pd

import scoring

print(f"Python {platform.python_version()} | numpy {np.__version__} | pandas {pd.__version__}")
with open("../data/processed/data.pkl", "rb") as f:   # Oscar's hand-over file (Oscar/make_handover.py)
    d = pickle.load(f)
df = scoring.frame(d)
INPUTS = {"Item 7": d["X"][:, 768:], "both": d["X"]}   # data.pkl X: mean Item 1 vector, then mean Item 7 vector
del d
x_all, y_all, yr_all = df["logvol_before"].to_numpy(), df["logvol_after"].to_numpy(), df["year"].to_numpy()
ALPHAS = np.logspace(0, 9, 37)                         # the grid of 02
print(f"{len(df):,} reports, inputs {[v.shape for v in INPUTS.values()]}")

Python 3.13.17 | numpy 2.5.3 | pandas 3.0.6


27,142 reports, inputs [(27142, 768), (27142, 1536)]


## 1. The fits

`centre` removes each year's own mean from a set of rows. Inputs are then standardised with the training rows'
within-year standard deviation, as `ridge_sums.py` does. The joint fit leaves $x$ unpenalised, which by
Frisch–Waugh–Lovell is the same as removing $x$ from both $y$ and every text column first, then running an ordinary
ridge on what is left; that is how it is solved here, with one eigen-decomposition serving every $\lambda$.

In [3]:
def centre(v, years):
    """Subtract each year's own mean (rows of v grouped by years)."""
    v = np.asarray(v, dtype=np.float64)
    out = v.copy()
    for yr in np.unique(years):
        m = years == yr
        out[m] = v[m] - v[m].mean(axis=0)
    return out

def ridge_path(Z, t, alphas):
    """Ridge coefficients of t on Z for every alpha (no intercept: everything is centred). Z is p columns."""
    eig, V = np.linalg.eigh(Z.T @ Z)
    q = V.T @ (Z.T @ t)
    return V @ (q[:, None] / (np.clip(eig, 0, None)[:, None] + np.asarray(alphas)[None, :]))   # p x len(alphas)

def fit(kind, X, rows, alphas, a=None, b=None):
    """One within-year fit on `rows`. kind: 'two-step', 'joint' or 'no text'.
    Returns the pieces needed to predict: sd, beta (text), gamma (slope on x) for each alpha."""
    yrs = yr_all[rows]
    xc, yc = centre(x_all[rows], yrs), centre(y_all[rows], yrs)
    if kind == "no text":
        return {"sd": None, "beta": None, "gamma": np.full(len(alphas), (xc @ yc) / (xc @ xc))}
    Xc = centre(X[rows], yrs)
    sd = np.sqrt((Xc ** 2).mean(axis=0)) + 1e-12
    Z = Xc / sd
    if kind == "two-step":                      # the text explains the line's within-year residual; slope fixed at b
        beta = ridge_path(Z, yc - b * xc, alphas)
        return {"sd": sd, "beta": beta, "gamma": np.full(len(alphas), b)}
    # joint: remove x from y and from every column of Z (Frisch-Waugh-Lovell), ridge on the rest, then recover gamma
    k = xc @ xc
    Zt, yt = Z - np.outer(xc, (xc @ Z) / k), yc - xc * (xc @ yc) / k
    beta = ridge_path(Zt, yt, alphas)
    gamma = (xc @ yc - (xc @ Z) @ beta) / k
    return {"sd": sd, "beta": beta, "gamma": gamma}

def within_part(model, X, rows, j):
    """The within-year prediction (centred on the rows' own year) for alpha index j."""
    yrs = yr_all[rows]
    p = model["gamma"][j] * centre(x_all[rows], yrs)
    if model["beta"] is not None:
        p = p + (centre(X[rows], yrs) / model["sd"]) @ model["beta"][:, j]
    return p

CHOSEN = []

def make_model(kind, label):
    X = INPUTS[label]
    def predict(df_, train, test):
        years = sorted(set(yr_all[train]))
        sub, val = np.flatnonzero(train & (yr_all != years[-1])), np.flatnonzero(train & (yr_all == years[-1]))
        tr, te = np.flatnonzero(train), np.flatnonzero(test)
        a, b = scoring.fit_line(df_, train)
        # choose lambda on the last training year (fit on the others, with their own line, as in ridge_sums.fit)
        a_s, b_s = scoring.fit_line(df_, train & (yr_all != years[-1]))
        m_s = fit(kind, X, sub, ALPHAS, a_s, b_s)
        y_val = centre(y_all[val], yr_all[val])
        n_alpha = 1 if kind == "no text" else len(ALPHAS)
        j = int(np.argmin([np.mean((y_val - within_part(m_s, X, val, j)) ** 2) for j in range(n_alpha)]))
        m = fit(kind, X, tr, ALPHAS[j:j + 1], a, b)
        line_level = np.mean(a + b * x_all[te])                     # the line's own level for the test year
        CHOSEN.append({"model": kind, "input": label, "test_year": int(yr_all[te[0]]), "train_years": f"{years[0]}-{years[-1]}",
                       "alpha": float(ALPHAS[j]) if kind != "no text" else np.nan,
                       "line_slope": b, "fitted_slope": float(m["gamma"][0])})
        return line_level + within_part(m, X, te, 0)
    return predict

## 2. Checks

The two-step version must reproduce 02's within-year ridge (same inputs, grid and rule for $\lambda$), and the joint
fit's text coefficients must match a direct solve of the full system with $x$ unpenalised.

In [4]:
# 1. the joint solve against the full (p+1) system on one window, for one alpha
train, test = scoring.masks(df, 2006, "strict")
rows, X = np.flatnonzero(train), INPUTS["Item 7"]
m = fit("joint", X, rows, [1e3])
yrs = yr_all[rows]
xc, yc, Xc = centre(x_all[rows], yrs), centre(y_all[rows], yrs), centre(X[rows], yrs)
Z = np.column_stack([Xc / m["sd"], xc])
P = np.eye(Z.shape[1]) * 1e3; P[-1, -1] = 0                         # penalise the text, not x
direct = np.linalg.solve(Z.T @ Z + P, Z.T @ yc)
assert np.allclose(direct[:-1], m["beta"][:, 0], atol=1e-6) and np.isclose(direct[-1], m["gamma"][0])
print("joint solve matches the direct (p+1) system")

# 2. the two-step version reproduces 02
t0 = time.perf_counter()
two_step = scoring.evaluate(df, make_model("two-step", "Item 7"), name="two-step within-year ridge, Item 7")
old = pd.read_csv("../data/output/alex-02-ridge-results.csv")
old = old[old.model == "within-year ridge, Item 7"].set_index(["window", "test_year"])["mse"]
gap = (two_step.set_index(["window", "test_year"])["mse"] - old).abs().max()
print(f"two-step vs 02's within-year ridge: largest MSE difference {gap:.1e} ({time.perf_counter() - t0:.0f}s)")
assert gap < 1e-6

joint solve matches the direct (p+1) system


two-step vs 02's within-year ridge: largest MSE difference 1.3e-07 (38s)


## 3. All fits

In [5]:
t0 = time.perf_counter()
CHOSEN.clear()
res = [two_step]
for label in INPUTS:
    for kind in ["two-step", "joint"]:
        if (kind, label) != ("two-step", "Item 7"):
            res.append(scoring.evaluate(df, make_model(kind, label), name=f"{kind} within-year ridge, {label}"))
res.append(scoring.evaluate(df, make_model("no text", "Item 7"), name="no-text within-year slope"))
results = pd.concat(res, ignore_index=True)
results.to_csv("../data/output/alex-02b-joint-ridge-results.csv", index=False)
pd.DataFrame(CHOSEN).to_csv("../data/output/alex-02b-joint-ridge-fits.csv", index=False)
print(f"{results['model'].nunique()} models x 12 splits in {time.perf_counter() - t0:.0f}s; "
      f"largest level change against the line: {(results['level'] - results['baseline_level']).abs().max():.1e}")

5 models x 12 splits in 222s; largest level change against the line: 2.4e-16


## 4. Results

The change in MSE against the straight line, averaged over the six test years; with every level equal to the line's,
it is all spread. For comparison, Louis's FinBERT network (single seeds, from his saved results) and the attention
network with a refit (`03b`).

In [6]:
def avg(d):
    return d.groupby(["model", "window"], sort=False)["change_pct"].mean().unstack().round(2)

summary = avg(results)
louis = pd.read_csv("../data/output/louis-05-wordcount-v2-results.csv")
louis = louis[louis.model.str.startswith("network: FinBERT + last year (Huber, refit), seed")]
louis = louis.assign(spread_pts=100 * (louis.spread - louis.baseline_spread) / louis.baseline_mse)
summary.loc["Louis FinBERT network, spread part only (single seeds)"] = louis.groupby("window")["spread_pts"].mean().round(2)
att = pd.read_csv("../data/output/alex-03b-refit-results.csv")
summary.loc["attention, Item 7, refit (03b)"] = att[att.model.str.contains(", refit, seed")].groupby("window")["change_pct"].mean().round(2)
display(summary)

display(results.pivot_table(index=["model", "window"], columns="test_year", values="change_pct", sort=False).round(2))

fits = pd.read_csv("../data/output/alex-02b-joint-ridge-fits.csv")
display(fits.groupby(["model", "input"])[["line_slope", "fitted_slope"]].mean().round(3))

window,paper,strict
model,,
"two-step within-year ridge, Item 7",-1.02,-0.03
"joint within-year ridge, Item 7",-3.82,-2.25
"two-step within-year ridge, both",-0.74,0.19
"joint within-year ridge, both",-3.76,-2.07
no-text within-year slope,-0.46,-0.04
"Louis FinBERT network, spread part only (single seeds)",-2.70,-1.26
"attention, Item 7, refit (03b)",-1.22,-0.52


test_year                                  2001  2002  2003  2004  2005  2006
model                              window                                    
two-step within-year ridge, Item 7 paper  -1.77 -5.65  1.05  0.64 -0.00 -0.42
                                   strict -1.65 -4.40 -0.22  6.06  0.02 -0.00
joint within-year ridge, Item 7    paper  -2.77 -6.51 -1.03 -5.49 -3.89 -3.19
                                   strict -2.42 -5.05 -1.21  1.16 -3.51 -2.49
two-step within-year ridge, both   paper  -1.27 -4.73  1.78  0.65 -0.00 -0.85
                                   strict -1.73 -3.54  0.02  6.62 -0.21 -0.00
joint within-year ridge, both      paper  -2.76 -6.35 -1.22 -4.85 -3.96 -3.40
                                   strict -2.10 -4.49 -0.90  1.37 -3.47 -2.83
no-text within-year slope          paper   0.04 -0.61  0.26 -1.31 -1.15 -0.02
                                   strict  0.07 -0.07  0.15  0.73 -0.96 -0.14

line_slope  fitted_slope
model    input                           
joint    Item 7       0.873         0.739
         both         0.873         0.718
no text  Item 7       0.873         0.876
two-step both         0.873         0.873